# 第 14 章 AD-AS モデルのシミュレーション: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/14-adas-simulation.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
a = 0.8          # AS 曲線の傾き
c = 0.3          # AD 曲線の傾き
sigma_u = 1.5    # 需要ショックの標準偏差
sigma_v = 1.2    # 供給ショックの標準偏差
u = 1.0          # 1 期間だけの需要ショック（%）
v = 0.0          # 1 期間だけの供給ショック（%）
periods_after = 20  # ショックのあと、何期間を見るか
n = 100          # ショックが毎期起きるシミュレーションの長さ
seed = 0         # 乱数の種

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## ショックへの反応

In [ ]:
import numpy as np

from lib.chapter13.adas import AdasParameters
from lib.chapter14.simulation import impulse_response, price_half_life, simulate

params = AdasParameters(a=a, c=c, sigma_u=sigma_u, sigma_v=sigma_v)
df = impulse_response(params, u=u, v=v, periods_after=periods_after)
shock = df.iloc[3]
after = df.iloc[4]
print(f"ショックの期: 物価 {shock['p']:.4f}、産出量 {shock['y']:.4f}")
print(f"次の期: 産出量 {after['y']:.4f}")
print(f"半減期: {price_half_life(params):.2f} 四半期")
simulation = simulate(params, n, np.random.default_rng(seed))
std_y = simulation["y"].std()
print(f"毎期ショックのシミュレーションの産出量の標準偏差: {std_y:.4f}")

## 図: ショックへの反応

In [ ]:
from lib.chapter14.plot import plot_impulse_response

plot_impulse_response(df)

## 図: 毎期ショックが起きるシミュレーション

In [ ]:
from lib.chapter14.plot import plot_simulation

plot_simulation(simulation)

## 試してみよう

- `u = 0.0`、`v = 1.0` にすると、供給ショックなので、ショックの期は、
  物価が 0.8065 に上がり、産出量は −0.2419 に下がります
- `a = 1.6` にすると、半減期は 3.22 四半期から 1.77 四半期に短くなり、
  物価は速く元に戻ります
- `seed = 1` にすると、毎期ショックのシミュレーションの産出量の標準偏差は、
  1.2153 から 1.1425 に変わります（種が変わると、乱数が変わります）